In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import ImageDataGenerator

(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Normalization

X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32")  / 255.0

# One-hot encode labels

num_classes = 10
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test  = keras.utils.to_categorical(y_test,  num_classes)

# Data Augmentation
# Applied ONLY to training data, never to test data
datagen = ImageDataGenerator(
    horizontal_flip=True,       # randomly flip images left-right
    width_shift_range=0.1,      # randomly shift horizontally by 10%
    height_shift_range=0.1,     # randomly shift vertically by 10%
    rotation_range=10,          # randomly rotate by up to 10 degrees
    zoom_range=0.1              # randomly zoom in by up to 10%
)

datagen.fit(X_train)


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step


In [ ]:
def build_base_model():
    model = keras.Sequential([

        #  Block 1
        layers.Input(shape=(32, 32, 3)),

        layers.Conv2D(
            filters=16,
            kernel_size=3,   # the filter size
            padding="valid",       # no zero-padding → output shrinks
            activation="relu"      # ReLU: f(x) = max(0, x)
        ),

        layers.Conv2D(
            filters=16,
            kernel_size=3,
            padding="valid",
            activation="relu"
        ),

        layers.MaxPooling2D(pool_size=(2, 2)),  # spatial dims ÷ 2

        # Block 2
        layers.Conv2D(
            filters=32,
            kernel_size=3,
            padding="valid",
            activation="relu"
        ),

        layers.Conv2D(
            filters=32,
            kernel_size=3,
            padding="valid",
            activation="relu"
        ),

        layers.MaxPooling2D(pool_size=(2, 2)),

        #  Classifier
        layers.Flatten(),

        layers.Dense(
            units=10,
            activation="softmax"   # outputs probabilities  to 1
        )
    ], name="base_cnn")

    return model

base_model = build_base_model()
base_model.summary()

Model: "base_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 30, 30, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 28, 28, 16)     │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 10, 10, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │         8,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,666 (96.35 KB)

 Trainable params: 24,666 (96.35 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
base_model = build_base_model()
#  Compile
base_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n📦 Training Base CNN...\n")

history_base = base_model.fit(
    X_train, y_train,
    batch_size=64,
    epochs=5,

    validation_data=(X_test, y_test)
)
#  Evaluate
test_loss, test_acc = base_model.evaluate(X_test, y_test, verbose=0)
print(f"\n Base Model — Test Accuracy: {test_acc * 100:.2f}%")


📦 Training Base CNN...

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.3989 - loss: 1.6590 - val_accuracy: 0.4966 - val_loss: 1.3985
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5263 - loss: 1.3222 - val_accuracy: 0.5668 - val_loss: 1.2322
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5881 - loss: 1.1723 - val_accuracy: 0.6064 - val_loss: 1.1222
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6228 - loss: 1.0824 - val_accuracy: 0.6226 - val_loss: 1.0881
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6482 - loss: 1.0085 - val_accuracy: 0.6273 - val_loss: 1.0707

 Base Model — Test Accuracy: 62.73%


In [ ]:
def build_final_model():
    model = keras.Sequential([

        layers.Input(shape=(32, 32, 3)),

        # Block 1
        layers.Conv2D(64, kernel_size=3, padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Conv2D(64, kernel_size=3, padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.2),

        # Block 2
        layers.Conv2D(128, kernel_size=3, padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Conv2D(128, kernel_size=3, padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.2),

        # ── Block 3 ──────────────────────────────────────────────────────────
        layers.Conv2D(256, kernel_size=3, padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Conv2D(256, kernel_size=3, padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.2),

        # ── Block 4 ──────────────────────────────────────────────────────────
        layers.Conv2D(512, kernel_size=3, padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Conv2D(512, kernel_size=3, padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.2),

        # ── Classifier ───────────────────────────────────────────────────────
        layers.GlobalAveragePooling2D(),

        layers.Dense(512, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(10, activation="softmax")

    ], name="final_cnn")

    return model


#  Learning Rate Schedule
# Total steps = epochs × steps_per_epoch
total_steps = 5 * (50000 // 128)

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.002,   # start aggressive
    decay_steps=total_steps,
    alpha=1e-6                     # minimum LR at the end
)

#  Build & Compile
final_model = build_final_model()

final_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

final_model.summary()

#  Train
print("\ Training Final CNN...\n")

history_final = final_model.fit(
    X_train, y_train,
    batch_size=128,                # larger batch = less noise = faster convergence
    epochs=5,
    validation_data=(X_test, y_test)
)

test_loss, test_acc = final_model.evaluate(X_test, y_test, verbose=0)
print(f"\n Final Model — Test Accuracy: {test_acc * 100:.2f}%")

<>:90: SyntaxWarning: invalid escape sequence '\ '
<>:90: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_2712/3683651034.py:90: SyntaxWarning: invalid escape sequence '\ '
  print("\ Training Final CNN...\n")


Model: "final_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_40 (Conv2D)              │ (None, 32, 32, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_33          │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_28 (Activation)      │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_41 (Conv2D)              │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_34          │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_29 (Activation)      │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_20 (MaxPooling2D) │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_42 (Conv2D)              │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_35          │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_30 (Activation)      │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_43 (Conv2D)              │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_36          │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_31 (Activation)      │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_21 (MaxPooling2D) │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_44 (Conv2D)              │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_37          │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_32 (Activation)      │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_45 (Conv2D)              │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_38          │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_33 (Activation)      │ (None, 8, 8, 256)      │             

 Total params: 4,962,890 (18.93 MB)

 Trainable params: 4,958,026 (18.91 MB)

 Non-trainable params: 4,864 (19.00 KB)

\ Training Final CNN...

Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 43s 74ms/step - accuracy: 0.4653 - loss: 1.6963 - val_accuracy: 0.2131 - val_loss: 2.8774
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 48ms/step - accuracy: 0.6913 - loss: 1.2029 - val_accuracy: 0.6207 - val_loss: 1.3797
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 48ms/step - accuracy: 0.7759 - loss: 1.0201 - val_accuracy: 0.7513 - val_loss: 1.0593
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 48ms/step - accuracy: 0.8314 - loss: 0.9003 - val_accuracy: 0.8322 - val_loss: 0.8942
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 18s 47ms/step - accuracy: 0.8679 - loss: 0.8277 - val_accuracy: 0.8496 - val_loss: 0.8582

 Final Model — Test Accuracy: 84.96%


In [ ]:
def build_final_model():
    model = keras.Sequential([

        layers.Input(shape=(32, 32, 3)),

        # Block 1
        layers.Conv2D(64, kernel_size=3, padding="same"), #=> to allowed the network to learn more complex feature maps cuz each filter learn one
       # and we change the padding to prevent the image to  shrink after convolution.
        layers.BatchNormalization(),##This prevents any single layer from becoming too "loud" (high values) or too "quiet"
        # (low values), which is why you can train this so fast withour aggressive learning rate 0.002
        layers.Activation("relu"),

        layers.Conv2D(64, kernel_size=3, padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.2),##base model underfit due to low capacity, adding more filters would cause it to overfit and dropout will  solve this

        # Block 2
        layers.Conv2D(128, kernel_size=3, padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Conv2D(128, kernel_size=3, padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.2),



        #  Classifier
        layers.GlobalAveragePooling2D(),
        #instead of "Flattening" the 8*8*128 block into a
        #massive 8,192-number vector(can cuz an overfitting), this layer takes the average of each 8*8 ->128 feature to enter the fully connect layer


        layers.Dense(10, activation="softmax")

    ], name="final_cnn")

    return model


#  Learning Rate Schedule
# Total steps = epochs × steps_per_epoch
total_steps = 5 * (50000 // 128)

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.002,   # start aggressive
    decay_steps=total_steps,
    alpha=1e-6                     # minimum LR at the end
)

#  Build & Compile
final_model = build_final_model()

final_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

final_model.summary()

#  Train
print("\ Training Final CNN...\n")

history_final = final_model.fit(
    X_train, y_train,
    batch_size=128,                # larger batch = less noise = faster convergence => cuz Adam optimizer will rin  much smoother and that will let us use aggresive learning rate
    epochs=5,
    validation_data=(X_test, y_test)
)

test_loss, test_acc = final_model.evaluate(X_test, y_test, verbose=0)
print(f"\n Final Model — Test Accuracy: {test_acc * 100:.2f}%")

<>:68: SyntaxWarning: invalid escape sequence '\ '
<>:68: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_21685/1033143283.py:68: SyntaxWarning: invalid escape sequence '\ '
  print("\ Training Final CNN...\n")


Model: "final_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 262,986 (1.00 MB)

 Trainable params: 262,218 (1.00 MB)

 Non-trainable params: 768 (3.00 KB)

\ Training Final CNN...

Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 25s 39ms/step - accuracy: 0.5108 - loss: 1.5784 - val_accuracy: 0.1757 - val_loss: 3.1362
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 0.6482 - loss: 1.3121 - val_accuracy: 0.5424 - val_loss: 1.5129
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 0.7067 - loss: 1.2013 - val_accuracy: 0.6365 - val_loss: 1.3244
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - accuracy: 0.7463 - loss: 1.1244 - val_accuracy: 0.7264 - val_loss: 1.1474
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 0.7701 - loss: 1.0806 - val_accuracy: 0.7593 - val_loss: 1.0904

 Final Model — Test Accuracy: 75.93%
